# StudyForge Week 0 — PDF → ChromaDB → RAG

All-CPU pipeline. The GPU droplet is Week 2. Set `LLM_API_KEY` in `.env` only when you want a generated answer or quiz; ingest and retrieval work without it.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from scripts.make_sample_pdf import build_pdf
from studyforge.config import get_settings
from studyforge.chunk import chunk_pages
from studyforge.pdf import extract_pages
from studyforge.rag import ask, ingest_pdf, quiz
from studyforge.store import reset_collection, retrieve

pdf = ROOT / "sample" / "rag_primer.pdf"
build_pdf(pdf)
print("sample PDF:", pdf)

## 1. Extract text with page numbers (PyMuPDF)

In [ ]:
pages = extract_pages(pdf)
for page in pages:
    print(f"--- p.{page.page} ({len(page.text)} chars) ---")
    print(page.text[:240], "...\n")

## 2. Chunk per page (citations stay honest)

In [ ]:
chunks = chunk_pages(pages, chunk_size=800, overlap=120)
print(f"{len(chunks)} chunks from {len(pages)} pages")
for chunk in chunks:
    print(f"{chunk.chunk_id}  p.{chunk.page_start}  {len(chunk.text)} chars")

## 3. Embed + store in ChromaDB (CPU, sentence-transformers)

In [ ]:
settings = get_settings()
reset_collection(settings)
stats = ingest_pdf(pdf, settings=settings)
print(stats)

hits = retrieve("What is retrieval-augmented generation?", k=3, settings=settings)
for hit in hits:
    print(f"p.{hit['page_start']}  dist={hit['distance']:.3f}")
    print(hit["text"][:220], "\n")

## 4. Ask with citations (needs `LLM_API_KEY`)

If the key is empty, `ask` prints the retrieved passages instead of calling an API.

In [ ]:
result = ask("What is RAG and why do answers cite pages?", mode="exam")
print(result["answer"])
print("\ncitations:", result["citations"])

In [ ]:
quiz_result = quiz(topic="embeddings and the MI300X")
if quiz_result.get("error"):
    print(quiz_result["error"])
else:
    for i, q in enumerate(quiz_result["questions"], start=1):
        print(f"{i}. {q.get('question')}")
        print("   answer:", q.get("answer"), " page:", q.get("source_page"))